In [1]:
import pandas as pd
import numpy as np

gva_df = pd.read_csv('gva_2025_predictions_estimates.csv')
test_df = pd.read_csv('test.csv')
train_df = pd.read_csv('train.csv')

In [2]:
base = pd.concat([train_df, test_df])[['LSOA21CD', 'MSOA21CD']].merge(
    gva_df[['LSOA21CD', 'GVA_2023_actual', 'GVA_2025_predicted']], on='LSOA21CD', how='inner')
base = base[(base['GVA_2023_actual'] > 0) & (base['GVA_2025_predicted'] > 0)].copy()
base['ratio'] = base['GVA_2025_predicted'] / base['GVA_2023_actual']
msoa_ratio = base.groupby('MSOA21CD')['ratio'].mean()
global_ratio = base['ratio'].mean()

In [3]:
gva_subset = gva_df[['LSOA21CD', 'GVA_2025_predicted', 'log_GVA_2025_predicted']]
test_updated = test_df.merge(gva_subset, on='LSOA21CD', how='left')
train_updated = train_df.merge(gva_subset, on='LSOA21CD', how='left')

In [4]:
def impute_gva_2025(df):
    df = df.copy()
    need = df['GVA_2025_predicted'].isna() | (df['GVA_2025_predicted'] <= 0)
    df['gva_2025_imputed'] = need.astype(int)

    gva_2023 = np.expm1(df.loc[need, 'log_total_GVA_2023'])
    ratio = df.loc[need, 'MSOA21CD'].map(msoa_ratio).fillna(global_ratio)
    df.loc[need, 'GVA_2025_predicted'] = gva_2023 * ratio

    log_na = df['log_GVA_2025_predicted'].isna()
    df.loc[log_na, 'log_GVA_2025_predicted'] = np.log1p(df.loc[log_na, 'GVA_2025_predicted'])
    return df

test_updated = impute_gva_2025(test_updated)
train_updated = impute_gva_2025(train_updated)

In [5]:
print(test_updated[['GVA_2025_predicted', 'log_GVA_2025_predicted']].isnull().sum())
print(train_updated[['GVA_2025_predicted', 'log_GVA_2025_predicted']].isnull().sum())

GVA_2025_predicted        0
log_GVA_2025_predicted    0
dtype: int64
GVA_2025_predicted        0
log_GVA_2025_predicted    0
dtype: int64


In [6]:
test_updated.to_csv('test_updated.csv', index=False)
train_updated.to_csv('train_updated.csv', index=False)